# Swiss Accidents

This notebook fetches ASTRA data containing info of all accidents with persons injured from [https://data.geo.admin.ch/](https://data.geo.admin.ch/browser/index.html#/collections/ch.astra.unfaelle-personenschaeden_alle/items/unfaelle-personenschaeden_alle?.language=de-CH&.asset=asset-unfaelle-personenschaeden_alle_2056-csv-zip).

The script 

1. downloads a multi-file zip (about 10 mb)
2. extracts the relevant csv, and
3. stores the result as a parquet file (and also to Excel)

In [1]:
import io
import zipfile

import polars as pl
import requests

## Fetch data

In [2]:
url = "https://data.geo.admin.ch/ch.astra.unfaelle-personenschaeden_alle/unfaelle-personenschaeden_alle/unfaelle-personenschaeden_alle_2056.csv.zip"
target_csv = "RoadTrafficAccidentLocations.csv"

In [9]:
response = requests.get(url, timeout=60)
response.raise_for_status()

with zipfile.ZipFile(io.BytesIO(response.content)) as archive:
    df = pl.read_csv(io.BytesIO(archive.read(target_csv)))

In [ ]:
print("Data shape:", df.shape)
df.head()

Data shape: (267761, 36)


AccidentUID,AccidentType,AccidentType_de,AccidentType_fr,AccidentType_it,AccidentType_en,AccidentSeverityCategory,AccidentSeverityCategory_de,AccidentSeverityCategory_fr,AccidentSeverityCategory_it,AccidentSeverityCategory_en,AccidentInvolvingPedestrian,AccidentInvolvingBicycle,AccidentInvolvingMotorcycle,RoadType,RoadType_de,RoadType_fr,RoadType_it,RoadType_en,AccidentLocation_CHLV95_E,AccidentLocation_CHLV95_N,CantonCode,MunicipalityCode,AccidentYear,AccidentMonth,AccidentMonth_de,AccidentMonth_fr,AccidentMonth_it,AccidentMonth_en,AccidentWeekDay,AccidentWeekDay_de,AccidentWeekDay_fr,AccidentWeekDay_it,AccidentWeekDay_en,AccidentHour,AccidentHour_text
str,str,str,str,str,str,str,str,str,str,str,bool,bool,bool,str,str,str,str,str,i64,i64,str,i64,i64,i64,str,str,str,str,str,str,str,str,str,i64,str
"""9965818AA56DF000E0430A865E33F0…","""at2""","""Auffahrunfall""","""accident par tamponnement""","""Incidente di tamponamento""","""Accident with rear-end collisi…","""as2""","""Unfall mit Schwerverletzten""","""accident avec blessés graves""","""Incidente con feriti gravi""","""Accident with severe injuries""",false,false,false,"""rt432""","""Hauptstrasse""","""route principale""","""Strada principale""","""Principal road""",2677632,1226133,"""ZG""",1702,2011,1,"""Januar""","""janvier""","""Gennaio""","""January""","""aw402""","""Dienstag""","""mardi""","""Martedì""","""Tuesday""",12,"""12h-13h"""
"""996651D38EC1A0C8E0430A865E33A0…","""at5""","""Überqueren der Fahrbahn""","""accident en traversant une rou…","""Incidente nell'attraversare la…","""Accident when crossing the lan…","""as3""","""Unfall mit Leichtverletzten""","""accident avec blessés légers""","""Incidente con feriti leggeri""","""Accident with light injuries""",false,false,false,"""rt432""","""Hauptstrasse""","""route principale""","""Strada principale""","""Principal road""",2718078,1273115,"""TG""",4831,2011,1,"""Januar""","""janvier""","""Gennaio""","""January""","""aw403""","""Mittwoch""","""mercredi""","""Mercoledì""","""Wednesday""",7,"""07h-08h"""
"""99797FE697E750A4E0430A865E3350…","""at4""","""Einbiegeunfall""","""accident en s'engageant sur un…","""Incidente nell’immettersi in u…","""Accident when turning-into mai…","""as2""","""Unfall mit Schwerverletzten""","""accident avec blessés graves""","""Incidente con feriti gravi""","""Accident with severe injuries""",false,true,false,"""rt432""","""Hauptstrasse""","""route principale""","""Strada principale""","""Principal road""",2680999,1225729,"""ZG""",1711,2011,1,"""Januar""","""janvier""","""Gennaio""","""January""","""aw402""","""Dienstag""","""mardi""","""Martedì""","""Tuesday""",11,"""11h-12h"""
"""997B044BBD5660F6E0430A865E3360…","""at8""","""Fussgängerunfall""","""accident impliquant des piéton…","""Incidente con pedoni""","""Accident involving pedestrian(…","""as3""","""Unfall mit Leichtverletzten""","""accident avec blessés légers""","""Incidente con feriti leggeri""","""Accident with light injuries""",true,false,false,"""rt433""","""Nebenstrasse""","""route secondaire""","""Strada secondaria""","""Minor road""",2681599,1225366,"""ZG""",1711,2011,1,"""Januar""","""janvier""","""Gennaio""","""January""","""aw401""","""Montag""","""lundi""","""Lunedí""","""Monday""",11,"""11h-12h"""
"""99AE12B0D5D93060E0430A865E3330…","""at4""","""Einbiegeunfall""","""accident en s'engageant sur un…","""Incidente nell’immettersi in u…","""Accident when turning-into mai…","""as3""","""Unfall mit Leichtverletzten""","""accident avec blessés légers""","""Incidente con feriti leggeri""","""Accident with light injuries""",false,false,false,"""rt432""","""Hauptstrasse""","""route principale""","""Strada principale""","""Principal road""",2646335,1244460,"""AG""",4010,2011,1,"""Januar""","""janvier""","""Gennaio""","""January""","""aw402""","""Dienstag""","""mardi""","""Martedì""","""Tuesday""",18,"""18h-19h"""


## Store as Parquet file

In [12]:
df.write_parquet("accidents.parquet")

## Store as Excel

In [13]:
columns_to_export = [
    "AccidentType_de",
    "AccidentSeverityCategory_de",
    "AccidentInvolvingPedestrian",
    "AccidentInvolvingBicycle",
    "AccidentInvolvingMotorcycle",
    "RoadType_de",
    "AccidentLocation_CHLV95_E",
    "AccidentLocation_CHLV95_N",
    "CantonCode",
    "MunicipalityCode",
    "AccidentYear",
    "AccidentMonth_de",
    "AccidentWeekDay_de",
]

df.select(columns_to_export).write_excel("accidents_selected.xlsx")
